In [1]:
import cv2
from PIL import Image
import torch
from transformers import BlipProcessor, BlipForConditionalGeneration
import speech_recognition as sr
import threading

# -------------------- LOAD MODEL --------------------
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# -------------------- AUDIO SETUP --------------------
recognizer = sr.Recognizer()
mic = sr.Microphone()

audio_text = "Listening..."

def listen_audio():
    global audio_text
    while True:
        try:
            with mic as source:
                recognizer.adjust_for_ambient_noise(source, duration=1)
                audio = recognizer.listen(source, phrase_time_limit=3)

            text = recognizer.recognize_google(audio)
            audio_text = text

        except:
            audio_text = "..."

# Start audio thread
threading.Thread(target=listen_audio, daemon=True).start()

# -------------------- VIDEO FILE --------------------
video_path = "sample_video.mp4"   # 👉 change this path
cap = cv2.VideoCapture(video_path)

frame_count = 0
caption = "Loading..."

print("Press 'q' or ESC to quit")

# -------------------- MAIN LOOP --------------------
while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.resize(frame, (320, 240))
    frame_count += 1

    # Generate caption every 30 frames
    if frame_count % 30 == 0:
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        pil_image = Image.fromarray(image)

        inputs = processor(images=pil_image, return_tensors="pt").to(device)

        with torch.no_grad():
            output = model.generate(**inputs, max_length=20)

        caption = processor.decode(output[0], skip_special_tokens=True)

    # Display captions
    cv2.putText(frame, "Video: " + caption, (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    cv2.putText(frame, "Audio: " + audio_text, (10, 60),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)

    cv2.imshow("Video + Audio Caption Generator", frame)

    key = cv2.waitKey(1)
    if key == ord('q') or key == 27:
        break

cap.release()
cv2.destroyAllWindows()

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identic

Press 'q' or ESC to quit
